In [0]:
#spark.sql("DROP TABLE IF EXISTS workspace.gold.fact_ventas")

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.gold.fact_ventas (
  id_venta LONG,
  id_tienda LONG,
  id_ciudad LONG,
  id_producto LONG,
  precio_unitario INT,
  unidades_vendidas INT,
  total_ventas INT,
  ganancia_operativa INT,
  id_canal_venta LONG,
  id_calendario LONG
)
""")

In [0]:
import pandas as pd

# Llamo a las tablas de dimensiones, ademas de la tabla de ventas
df = spark.table('workspace.silver.adidas_us_sales').toPandas()
dim_ciudades = spark.table('workspace.gold.dim_ciudades').toPandas()
dim_canal_venta = spark.table('workspace.gold.dim_canal_venta').toPandas()
dim_tiendas = spark.table('workspace.gold.dim_tiendas').toPandas()
dim_productos = spark.table('workspace.gold.dim_productos').toPandas()
dim_calendario = spark.table('workspace.gold.dim_calendario').toPandas()

#Se hace left join para unir las tablas y llevas las llaves de las dimensiones
df_fact = df.merge(
    dim_ciudades,
    left_on=['ciudad', 'estado', 'region'],
    right_on=['nombre_ciudad', 'estado_ciudad', 'region_ciudad'],
    how='left'
)

df_fact = df_fact.merge(
    dim_canal_venta,
    left_on='canal_venta',
    right_on='nombre_canal_venta',
    how='left'
)

df_fact = df_fact.merge(
    dim_tiendas,
    left_on='tienda',
    right_on='nombre_tienda',
    how = 'left'
)

df_fact = df_fact.merge(
    dim_productos,
    left_on='producto',
    right_on='nombre_producto',
    how='left'
)

df_fact = df_fact.merge(
    dim_calendario[['id_calendario', 'fecha']],
    left_on='fecha_factura',
    right_on='fecha',
    how='left'
)

#Se eliminan las tablas innecesarias después de los joins
df_fact = df_fact.drop(
    columns = {'tienda', 'region', 'fecha_factura', 'estado', 'ciudad', 'producto', 'canal_venta', 'nombre_ciudad', 'estado_ciudad', 'region_ciudad', 'ciudad_estado', 'nombre_canal_venta', 'nombre_tienda', 'nombre_producto', 'fecha'}
)

df_spark = spark.createDataFrame(df_fact)

In [0]:
# Usamos overwrite para reemplazar completamente los datos existentes
df_spark.write.format("delta") \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.fact_ventas")

In [0]:
%sql
--select COUNT(*) from workspace.gold.fact_ventas